In [ ]:
# Cell 1 - run information (required)
import os, subprocess, sys
BUNDLE = "/content/p2_bundle"
if os.path.isdir("/content"):                      # Colab: everything comes from p2_bundle.zip on Drive
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir(BUNDLE):
        subprocess.run(["unzip", "-q", "/content/drive/MyDrive/genai_lab2/p2_bundle.zip", "-d", "/content"],
                       check=True)
    os.chdir(f"{BUNDLE}/data266-lab2-team32")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import subprocess, sys, random, numpy as np, torch
SEED = 266; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
text=True).stdout.strip()
print({"commit": commit, "python": sys.version.split()[0], "torch": torch.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu", "seed": SEED})

# Part 2 - Fine-tuning Stable Diffusion 1.5 on Naruto images

DATA 266 Lab 2, Pair_Programming_Team_32. Runs on one Colab T4. Code is in `part2_diffusion/src/`,
settings in `part2_diffusion/configs/config.yaml`.

Everything (packages, dataset, SD 1.5, CLIP, metric weights) comes from `p2_bundle.zip` in Google
Drive; nothing is downloaded. Results go to `MyDrive/data266_p2`.

Set `TEST_RUN = True` first: 10 training steps and 3 prompts, about 15 minutes, to check that
everything works. Its files go to `MyDrive/data266_p2_test`. Then set it back to `False`.

In [ ]:
TEST_RUN = False

if os.path.isdir("/content"):
    # packages, dataset, models and metric weights all come from the bundle: no downloads
    !pip install -q --no-index --no-deps {BUNDLE}/wheels/*.whl
    results = "/content/drive/MyDrive/genai_lab2/data266_p2" + ("_test" if TEST_RUN else "")
    os.environ.update(HF_HUB_OFFLINE="1", TORCH_HOME=f"{BUNDLE}/torch",
                      P2_RAW=f"{BUNDLE}/raw/data", P2_MODELS=f"{BUNDLE}/models",
                      P2_DATA="/content/data", P2_RESULTS=results, P2_BIG=f"{results}/big")

from importlib.metadata import version
for pkg in ["torch", "diffusers", "transformers", "peft", "datasets", "torchmetrics", "lpips"]:
    print(f"{pkg}=={version(pkg)}")

## 2.1 Data preparation
Remove corrupted images (cannot be decoded), exact duplicates (same MD5) and near duplicates
(dHash distance of 4 bits or less); split 90/10 with seed 266; resize to 512x512.

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from part2_diffusion.src import common, data

summary = data.prepare()
print(json.dumps(summary, indent=2))

train_rows = [json.loads(line) for line in open(common.DATA / "train" / "metadata.jsonl")][:6]
fig, axes = plt.subplots(1, 6, figsize=(18, 3.6))
for ax, row in zip(axes, train_rows):
    ax.imshow(Image.open(common.DATA / "train" / row["file_name"]))
    ax.set_title(row["text"][:40], fontsize=8)
    ax.axis("off")
plt.show()

In [ ]:
prompts = common.load_prompts()
if TEST_RUN:   # one prompt per category
    prompts = [next(p for p in prompts if p["category"] == c)
               for c in ("in_domain", "out_of_domain", "compositional")]
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(prompts)

## 2.2 Fine-tuning
Two runs with the same data and step budget: LoRA rank 4 and LoRA rank 32. A run that has already
finished (weights on Drive) is not trained again.

In [ ]:
from part2_diffusion.src import train
steps = 10 if TEST_RUN else None
stats = {run: train.train(run, max_steps=steps) for run in common.RUNS}

In [ ]:
cols = ["rank", "trainable_params", "learning_rate", "steps", "batch_size",
        "peak_gpu_memory_gb", "training_minutes", "gpu_hours", "gpu"]
display(pd.DataFrame(stats).T[cols])

plt.figure(figsize=(8, 3.5))
for run, s in stats.items():
    smooth = pd.Series(s["losses"]).rolling(50, min_periods=1).mean()
    plt.plot(smooth, label=run, linewidth=1.5)
plt.xlabel("step"); plt.ylabel("training loss (50-step average)"); plt.legend()
plt.savefig(common.OUTPUTS / "loss_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## 2.3 Image generation
Every prompt with seeds 0 and 1 for the base model and both runs, all with the same sampler
(DPM-Solver++, 30 steps, guidance 7.5). The pipeline's safety checker is on.

In [ ]:
from part2_diffusion.src import generate
gens = generate.generate(prompts)
print(len(gens), "images,", gens["nsfw_flagged"].sum(), "flagged by the NSFW filter")

In [ ]:
grid_ids = ["ID01", "ID02", "ID03", "ID04", "OD01", "OD02", "CP01", "CP02"]
(common.OUTPUTS / "grids").mkdir(parents=True, exist_ok=True)
for p in prompts:
    if p["prompt_id"] not in grid_ids:
        continue
    fig, axes = plt.subplots(2, 3, figsize=(9, 6.4))
    for col, model in enumerate(common.MODELS):
        for row, seed in enumerate((0, 1)):
            axes[row, col].imshow(Image.open(common.image_path(model, p["prompt_id"], seed)))
            axes[row, col].set_title(f"{model}, seed {seed}", fontsize=9)
            axes[row, col].axis("off")
    fig.suptitle(f"{p['prompt_id']}: {p['prompt']}", fontsize=10)
    fig.savefig(common.OUTPUTS / "grids" / f"{p['prompt_id']}.jpg", dpi=80, bbox_inches="tight")
    plt.show()

## 2.4 Evaluation
Primary: KID (in-domain generations vs test images; noisy with only 20 images) and CLIP score.
Secondary: LPIPS between the two seeds of each prompt (diversity).

In [ ]:
from part2_diffusion.src import evaluate
metrics, clip, clip_df = evaluate.evaluate(prompts)
display(metrics.round(4))
display(clip_df.groupby(["model", "category"])["clip_score"].mean().unstack().round(2))

In [ ]:
# Final run: lower KID. Change this if the grids and the human vote point the other way.
FINAL_RUN = metrics.loc[common.RUNS, "kid_mean"].idxmin()
print("final run:", FINAL_RUN)

### Memorization check
For 10 in-domain images of the final run (seed 0), the most similar training image by CLIP
image similarity. The 3 closest pairs are shown.

In [ ]:
mem = evaluate.memorization(clip, FINAL_RUN, prompts)
mem.to_csv(common.METRICS / "memorization.csv", index=False)
display(mem)

fig, axes = plt.subplots(3, 2, figsize=(6, 9))
for row, (_, r) in enumerate(mem.head(3).iterrows()):
    axes[row, 0].imshow(Image.open(common.image_path(FINAL_RUN, r["prompt_id"], 0)))
    axes[row, 0].set_title(f"generated {r['prompt_id']}", fontsize=9)
    axes[row, 1].imshow(Image.open(common.DATA / "train" / r["closest_train_image"]))
    axes[row, 1].set_title(f"closest training image, similarity {r['similarity']:.3f}", fontsize=9)
    for ax in axes[row]: ax.axis("off")
fig.savefig(common.OUTPUTS / "memorization_top3.jpg", dpi=80, bbox_inches="tight")
plt.show()

### Failure modes
The final run's lowest CLIP-score images (where it ignored the prompt), to pick two failure
modes for the report.

In [ ]:
worst = clip_df[clip_df.model == FINAL_RUN].nsmallest(4, "clip_score")
fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
text = {p["prompt_id"]: p["prompt"] for p in prompts}
for ax, (_, r) in zip(axes, worst.iterrows()):
    ax.imshow(Image.open(common.image_path(FINAL_RUN, r["prompt_id"], r["seed"])))
    ax.set_title(f"{r['prompt_id']} seed {r['seed']}, CLIP {r['clip_score']:.1f}\n{text[r['prompt_id']][:45]}", fontsize=8)
    ax.axis("off")
fig.savefig(common.OUTPUTS / "failure_examples.jpg", dpi=80, bbox_inches="tight")
plt.show()

### Human evaluation
25 blinded pairs, final run vs base model. Raters look at `outputs/human_eval/pair_XX.jpg` (in `MyDrive/data266_p2`),
copy `ratings_TEMPLATE.csv` to `metrics/human_eval/ratings_<name>.csv` and write A or B in both columns.
Once the three files are there, run this notebook again (finished training is skipped).

In [ ]:
from part2_diffusion.src import human_eval
human_eval.make_pairs(FINAL_RUN)
result = human_eval.score(FINAL_RUN)
if result is not None:
    win_rates, per_rater = result
    display(win_rates.round(3)); display(per_rater.round(3))

## Results

In [ ]:
table = metrics.copy()
table["gpu_hours"] = [stats[m]["gpu_hours"] if m in stats else None for m in table.index]
table["peak_gpu_memory_gb"] = [stats[m]["peak_gpu_memory_gb"] if m in stats else None for m in table.index]
if result is not None:
    table.loc[FINAL_RUN, "human_win_rate_style"] = win_rates.loc["better_naruto_style", "final_run_win_rate"]
    table.loc[FINAL_RUN, "human_win_rate_prompt"] = win_rates.loc["better_prompt_match", "final_run_win_rate"]
table.to_csv(common.METRICS / "results_table.csv")
table.round(4)